In [4]:
import tkinter as tk
from tkinter import messagebox
import time

In [5]:
class TicTacToeAI:
    def __init__(self, root):
        self.root = root
        self.root.title("Tic-Tac-Toe AI: Minimax, Alpha-Beta, Expectimax")
        self.root.geometry("600x450")
        
        # Biến trạng thái trò chơi
        self.board = [' ' for _ in range(9)]
        self.human_player = 'X'
        self.ai_player = 'O'
        self.nodes_evaluated = 0
        
        self.create_widgets()
        self.log_message("Hệ thống khởi động. Bạn là 'X', AI là 'O'. Mời bạn đi trước!")

    def create_widgets(self):
        # Khung chứa bảng cờ
        self.board_frame = tk.Frame(self.root)
        self.board_frame.pack(side=tk.LEFT, padx=20, pady=20)
        
        self.buttons = []
        for i in range(9):
            btn = tk.Button(self.board_frame, text=' ', font=('Arial', 24, 'bold'), width=5, height=2,
                            command=lambda i=i: self.human_move(i))
            btn.grid(row=i//3, column=i%3)
            self.buttons.append(btn)
            
        # Khung điều khiển và Log
        self.control_frame = tk.Frame(self.root)
        self.control_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=20, pady=20)
        
        tk.Label(self.control_frame, text="Chọn thuật toán AI:", font=('Arial', 12, 'bold')).pack(anchor=tk.W)
        
        self.algorithm_var = tk.StringVar()
        self.algorithm_var.set("Alpha-Beta") # Default
        
        tk.Radiobutton(self.control_frame, text="Minimax (Vét cạn)", variable=self.algorithm_var, value="Minimax").pack(anchor=tk.W)
        tk.Radiobutton(self.control_frame, text="Alpha-Beta (Cắt tỉa)", variable=self.algorithm_var, value="Alpha-Beta").pack(anchor=tk.W)
        tk.Radiobutton(self.control_frame, text="Expectimax (Xác suất)", variable=self.algorithm_var, value="Expectimax").pack(anchor=tk.W)
        
        tk.Button(self.control_frame, text="Khởi động lại", command=self.reset_game, bg="lightcoral").pack(pady=10, fill=tk.X)
        
        tk.Label(self.control_frame, text="Log hoạt động:", font=('Arial', 12, 'bold')).pack(anchor=tk.W)
        self.log_text = tk.Text(self.control_frame, height=12, width=30, font=('Consolas', 9))
        self.log_text.pack(fill=tk.BOTH, expand=True)

    def log_message(self, message):
        self.log_text.insert(tk.END, message + "\n")
        self.log_text.see(tk.END)

    def reset_game(self):
        self.board = [' ' for _ in range(9)]
        for btn in self.buttons:
            btn.config(text=' ', state=tk.NORMAL)
        self.log_message("-" * 20)
        self.log_message("Đã làm mới bàn cờ. Mời 'X' đi trước.")

    def check_winner(self, board, player):
        win_conditions = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8], # Ngang
            [0, 3, 6], [1, 4, 7], [2, 5, 8], # Dọc
            [0, 4, 8], [2, 4, 6]             # Chéo
        ]
        return any(all(board[i] == player for i in condition) for condition in win_conditions)

    def is_board_full(self, board):
        return ' ' not in board

    def human_move(self, index):
        if self.board[index] == ' ':
            self.board[index] = self.human_player
            self.buttons[index].config(text=self.human_player, fg="blue")
            
            if self.check_winner(self.board, self.human_player):
                self.log_message("Bạn (X) đã thắng!")
                self.game_over()
                return
            if self.is_board_full(self.board):
                self.log_message("Hòa!")
                self.game_over()
                return
                
            self.root.update()
            self.ai_move()

    def game_over(self):
        for btn in self.buttons:
            btn.config(state=tk.DISABLED)

    # ---------------------------------------------------------
    # PHẦN THUẬT TOÁN AI
    # ---------------------------------------------------------
    def evaluate(self, board):
        if self.check_winner(board, self.ai_player): return 10
        elif self.check_winner(board, self.human_player): return -10
        else: return 0

    def minimax(self, board, depth, is_maximizing):
        self.nodes_evaluated += 1
        score = self.evaluate(board)
        if score == 10: return score - depth
        if score == -10: return score + depth
        if self.is_board_full(board): return 0

        if is_maximizing:
            best = -float('inf')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.ai_player
                    best = max(best, self.minimax(board, depth + 1, not is_maximizing))
                    board[i] = ' '
            return best
        else:
            best = float('inf')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.human_player
                    best = min(best, self.minimax(board, depth + 1, not is_maximizing))
                    board[i] = ' '
            return best

    def alphabeta(self, board, depth, alpha, beta, is_maximizing):
        self.nodes_evaluated += 1
        score = self.evaluate(board)
        if score == 10: return score - depth
        if score == -10: return score + depth
        if self.is_board_full(board): return 0

        if is_maximizing:
            best = -float('inf')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.ai_player
                    best = max(best, self.alphabeta(board, depth + 1, alpha, beta, not is_maximizing))
                    board[i] = ' '
                    alpha = max(alpha, best)
                    if beta <= alpha: break # Cắt tỉa nhánh Beta
            return best
        else:
            best = float('inf')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.human_player
                    best = min(best, self.alphabeta(board, depth + 1, alpha, beta, not is_maximizing))
                    board[i] = ' '
                    beta = min(beta, best)
                    if beta <= alpha: break # Cắt tỉa nhánh Alpha
            return best

    def expectimax(self, board, depth, is_maximizing):
        self.nodes_evaluated += 1
        score = self.evaluate(board)
        if score == 10: return score - depth
        if score == -10: return score + depth
        if self.is_board_full(board): return 0

        if is_maximizing:
            best = -float('inf')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.ai_player
                    best = max(best, self.expectimax(board, depth + 1, not is_maximizing))
                    board[i] = ' '
            return best
        else:
            # Chance node: Trả về trung bình (Average) thay vì Min
            expected_value = 0
            empty_spots = board.count(' ')
            for i in range(9):
                if board[i] == ' ':
                    board[i] = self.human_player
                    expected_value += self.expectimax(board, depth + 1, not is_maximizing)
                    board[i] = ' '
            return expected_value / empty_spots

    def ai_move(self):
        algo = self.algorithm_var.get()
        self.log_message(f"AI đang nghĩ bằng {algo}...")
        self.nodes_evaluated = 0
        start_time = time.time()
        
        best_val = -float('inf')
        best_move = -1
        
        for i in range(9):
            if self.board[i] == ' ':
                self.board[i] = self.ai_player
                
                if algo == "Minimax":
                    move_val = self.minimax(self.board, 0, False)
                elif algo == "Alpha-Beta":
                    move_val = self.alphabeta(self.board, 0, -float('inf'), float('inf'), False)
                elif algo == "Expectimax":
                    move_val = self.expectimax(self.board, 0, False)
                    
                self.board[i] = ' '
                
                if move_val > best_val:
                    best_move = i
                    best_val = move_val

        elapsed_time = (time.time() - start_time) * 1000
        
        # Cập nhật bàn cờ
        self.board[best_move] = self.ai_player
        self.buttons[best_move].config(text=self.ai_player, fg="red")
        
        # Ghi Log
        self.log_message(f"-> Chọn ô số {best_move}")
        self.log_message(f"-> Nodes duyệt: {self.nodes_evaluated}")
        self.log_message(f"-> Thời gian: {elapsed_time:.2f} ms")
        self.log_message("-" * 20)
        
        if self.check_winner(self.board, self.ai_player):
            self.log_message("AI (O) đã thắng!")
            self.game_over()
        elif self.is_board_full(self.board):
            self.log_message("Hòa!")
            self.game_over()

In [6]:
if __name__ == "__main__":
    root = tk.Tk()
    app = TicTacToeAI(root)
    root.mainloop()